Train an nnU-Net v2 model on 6 chest classes (Right Lung, Left Lung, Heart,
Trachea, Aorta, Spine), using pseudo-labels distilled from TotalSegmentator.

Run inside a Kaggle notebook with GPU (Settings > Accelerator > GPU T4 x2) and
internet on (Settings > Internet > On) — this pulls chest CT volumes from the
Medical Segmentation Decathlon and downloads TotalSegmentator's weights.

This trains a STUDENT of TotalSegmentator: it can only approach
TotalSegmentator's accuracy on these pseudo-labels, not exceed it, until some
predictions are hand-corrected and fed back in with finetune_chest_nnunet.py.
See build_pseudo_labels.py and the project README for that loop.

[Cell 1] Install nnU-Net v2 and TotalSegmentator

In [ ]:
!pip install -q nnunetv2 TotalSegmentator

[Cell 2] Paths, dataset id, and case budget

In [ ]:
# /kaggle/input is read-only, so nnU-Net's raw/preprocessed/results dirs must
# live under /kaggle/working (the only writable, persisted-as-output location).
import os
from pathlib import Path

WORK = Path("/kaggle/working")
TASK06_DIR = WORK / "Task06_Lung"  # extracted MSD tarball; volumes under imagesTr/
DATASET_ID = "501"
DATASET_NAME = f"Dataset{DATASET_ID}_ChestCT"
CONFIGURATION = "3d_fullres"  # trachea/aorta are thin, continuous structures that
                              # 2d slicing fragments; keep 3D context.
TRAINER = "nnUNetTrainer_100epochs"  # drop to _50epochs/_20epochs if the session times out

os.environ["nnUNet_raw"] = str(WORK / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(WORK / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(WORK / "nnUNet_results")
for path in (os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]):
    Path(path).mkdir(parents=True, exist_ok=True)

# Each CT volume is tens of MB and TotalSegmentator inference is the slow part
# (~1-2 min/case on a T4). Start small to prove the pipeline runs end to end;
# raise once that succeeds (Task06_Lung has 63 cases total, watch disk/time).
MAX_CASES = 10  # ponytail: raise (up to 63) once a small run succeeds

[Cell 3] Fetch chest CT volumes from the Medical Segmentation Decathlon

In [ ]:
# Task06_Lung is a public, no-auth tarball of 63 chest CT volumes (its own
# tumor labels are ignored here — this pipeline pseudo-labels its images with
# TotalSegmentator instead). Source: http://medicaldecathlon.com/, mirrored at
# the S3 bucket below (verified reachable 2026-08).
import tarfile
import requests
from tqdm import tqdm

TASK06_URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"
tar_path = WORK / "Task06_Lung.tar"

if not TASK06_DIR.exists():
    if not tar_path.exists():
        response = requests.get(TASK06_URL, stream=True, timeout=100)
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with tar_path.open("wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="Task06_Lung.tar") as bar:
            for chunk in response.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))

    # Extract only imagesTr/ and only the first MAX_CASES cases actually used,
    # not the full tar (labelsTr/ and the extra cases would waste disk).
    with tarfile.open(tar_path) as tf:
        wanted_prefix = "Task06_Lung/imagesTr/"
        members = sorted(
            (m for m in tf.getmembers() if m.name.startswith(wanted_prefix) and m.name.endswith(".nii.gz")
             and "._" not in m.name),  # MSD tarballs include macOS AppleDouble junk files
            key=lambda m: m.name,
        )[:MAX_CASES]
        tf.extractall(WORK, members=members)

tar_path.unlink(missing_ok=True)  # multi-GB; /kaggle/working only gets ~20 GB
usage = __import__("shutil").disk_usage(WORK)
print(f"Disk: {usage.used / 2**30:.1f} GiB used, {usage.free / 2**30:.1f} GiB free")

[Cell 4] Build 6-class pseudo-labels with TotalSegmentator

In [ ]:
# Reuses build_pseudo_labels.py (this repo) rather than reimplementing the
# TotalSegmentator call + label remap: clone once, add to path, then call its
# CLI. It writes imagesTr/labelsTr/dataset.json directly in nnU-Net's layout,
# and drops any case whose field of view misses >1 of the 6 classes.
import subprocess
import sys

REPO_DIR = WORK / "Abdominal_segmentation"
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/<your-org>/Abdominal_segmentation.git", str(REPO_DIR)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR))

raw_dataset_dir = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
images_dir = TASK06_DIR / "imagesTr"

subprocess.run(
    [
        sys.executable, str(REPO_DIR / "build_pseudo_labels.py"),
        "--input", str(images_dir),
        "--output", str(raw_dataset_dir),
        "--device", "cuda",
    ],
    check=True,
)

[Cell 5] Preprocess

In [ ]:
# subprocess.run(check=True) is used instead of `!shell` so a failure here
# raises and stops the script, instead of silently falling through into
# training on whatever partial data happened to finish preprocessing.
import shutil
import subprocess

preprocessed_dir = Path(os.environ["nnUNet_preprocessed"]) / DATASET_NAME
if preprocessed_dir.exists():
    shutil.rmtree(preprocessed_dir)  # drop any stale/partial output from a prior failed run

subprocess.run(
    [
        "nnUNetv2_plan_and_preprocess", "-d", DATASET_ID,
        "-c", CONFIGURATION, "-np", "2",
        "--verify_dataset_integrity",
    ],
    check=True,
)

[Cell 6] Train folds 0 and 1 in parallel, one per T4

In [ ]:
# nnU-Net v2 has no built-in DDP for a single fold, so the productive use of a
# 2-GPU session is training two folds concurrently instead of one fold alone.
import os
import subprocess

procs = []
for fold, gpu_id in enumerate([0, 1]):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    procs.append(subprocess.Popen(
        ["nnUNetv2_train", DATASET_ID, CONFIGURATION, str(fold), "-tr", TRAINER],
        env=env,
    ))

for p in procs:
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"nnUNetv2_train exited with code {p.returncode}")

[Cell 7] Copy the trained model into Kaggle's persisted notebook output

In [ ]:
import shutil

output_model_dir = WORK / "trained_model"
shutil.copytree(
    Path(os.environ["nnUNet_results"]) / DATASET_NAME,
    output_model_dir,
    dirs_exist_ok=True,
)
print(f"Model saved to {output_model_dir} — download it from the notebook's Output tab.")
print("Run chest_ct_annotation.py with --backend nnunet --nnunet-model-dir "
      f"{output_model_dir}/{TRAINER}__nnUNetPlans__{CONFIGURATION}")

[Cell 8] Per-class Dice from nnU-Net's own cross-validation

In [ ]:
# nnUNetv2_train writes a 5-fold-style summary per trained fold with per-class
# Dice against its own held-out validation split of the pseudo-labels — no
# extra scoring code needed. Read fold 0's summary as a first check.
import json

summary_path = (
    Path(os.environ["nnUNet_results"]) / DATASET_NAME
    / f"{TRAINER}__nnUNetPlans__{CONFIGURATION}" / "fold_0" / "validation" / "summary.json"
)
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(f"{'class':<12}{'Dice':>8}")
    for label, stats in summary["mean"].items():
        print(f"{label:<12}{stats['Dice']:>8.3f}")
else:
    print(f"No summary yet at {summary_path} — did Cell 6 finish?")